Exercise task 5: Feature importance

Part A:

Just copying the lectures notebook code for tokenizer:

In [ ]:
import datasets
import evaluate
import transformers
import torch
import numpy as np
from pprint import pprint
from sklearn import metrics

In [ ]:
dset=datasets.load_dataset("imdb")
pprint(dset)
dset=dset.shuffle()
del dset["unsupervised"]
pprint(dset['train'][0]['text'])
print(dset['train'][0]['label'])

Tokenizer:

In [ ]:
base_tokenizer = transformers.AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
tokenizer = base_tokenizer.train_new_from_iterator(
    dset["train"]["text"],
    vocab_size=15000
)

In [ ]:
def encode(examples):
    return tokenizer(examples['text'],
                     #truncation=True,
                     #max_length=256
                     )

dset_tokenized = dset.map(encode,batched=True,num_proc=4)

for key,val in dset_tokenized["train"][0].items():
    print(key,":",val)

In [ ]:
collator=transformers.DataCollatorWithPadding(tokenizer)
small_data=[tokenizer("Hi there"), tokenizer("A little longer text!")]
print("small_data:\n")
pprint(small_data)
print("\n\ncollated:\n")
small_batch=collator(small_data)
pprint(small_batch)


In [ ]:
# A Transformers library model wants a config,
# I can simply inherit from the base
# class for pretrained configs
# We don't really need to do anything very special
class MLPConfig(transformers.PretrainedConfig):
    pass

# This is the model
class MLP(transformers.PreTrainedModel):

    config_class=MLPConfig

    # In the initialization method, one instantiates the layers
    # these will be, for the most part the trained parameters of the model
    def __init__(self,config):
        super().__init__(config)
        self.all_tied_weights_keys = {} #Annoying bug in Transformers, must have this here or else we crash on saved model load
        #### HERE WE CREATE THE MODEL'S LAYERS:
        self.vocab_size=config.vocab_size #embedding matrix row count
        # Build and initialize embedding of vocab size x hidden size
        assert tokenizer.vocab["[PAD]"]==0 #let's make sure our assumption of pad==0 holds!

        self.embedding=torch.nn.Embedding(num_embeddings=self.vocab_size,embedding_dim=config.hidden_size,padding_idx=0)
        # Initialize the embeddings to random values
        # Note! This function is relatively clever and keeps the embedding for 0, the padding, pure zeros
        torch.nn.init.uniform_(self.embedding.weight.data,-0.001,0.001) #initialize the embeddings with small random values

        # This takes care of the lower half of the network, now the upper half
        # Output layer: hidden size x output size
        self.output=torch.nn.Linear(in_features=config.hidden_size,out_features=config.nlabels)
        # Now we have the parameters of the model
        self.loss=torch.nn.CrossEntropyLoss() #This loss is meant for classification, so let's use it


    # The computation of the model is put into the forward() function
    # it receives a batch of data and optionally the correct `labels`
    #
    # If given `labels` we should return (loss,output)
    # if not, then we should return (output,)
    # that way the model can be used both for training and for inference
    def forward(self,input_ids,labels=None,**kwargs):
        #1) sum up the embeddings of the items
        embedded=self.embedding(input_ids) #(batch,ids)->(batch,ids,embedding_dim)
        # Since the Embedding keeps the first row of the matrix pure zeros, we don't need to worry about the padding 0 index
        # so next we sum the embeddings across the word dimension
        # (batch,ids,embedding_dim) -> (batch,embedding_dim)
        embedded_summed=torch.sum(embedded,dim=1)

        #2) apply non-linearity
        # (batch,embedding_dim) -> (batch,embedding_dim)
        projected=torch.tanh(embedded_summed) #Note how non-linearity is applied here and not when configuring the layer in __init__()

        #3) and now apply the upper, output layer of the network
        # (batch,embedding_dim) -> (batch, num_of_classes i.e. 2 in our case)
        logits=self.output(projected)

        # ...and that's all there is to it!

        #print("input_ids.shape",input_ids.shape)
        #print("embedded.shape",embedded.shape)
        #print("embedded_summed.shape",embedded_summed.shape)
        #print("projected.shape",projected.shape)
        #print("logits.shape",logits.shape)

        # If we have labels, we ought to calculate the loss
        if labels is not None:
            # You run the loss as loss(model_output,correct_labels)
            return (self.loss(logits,labels),logits) #2-tuple, i.e. pair of values returned
        else:
            # No labels, so just return the logits
            return (logits,) #this weird syntax means 1-tuple



In [ ]:
# Configure the model:
#   these parameters are used in the model's __init__()
mlp_config=MLPConfig(vocab_size=tokenizer.vocab_size,hidden_size=20,nlabels=2)
print("mlp config:", mlp_config)

# And now we can instantiate it
mlp=MLP(mlp_config)
print("mlp",mlp)
#we can make a little test with the small test batch we made earlier
#since it has no true labels, it should return a 1-tuple, which it will
out=mlp(input_ids=small_batch["input_ids"])
print("Output on one batch:",out)

In [ ]:
trainer_args = transformers.TrainingArguments(
    "mlp_checkpoints", #save checkpoints here
    eval_strategy="steps", #...and not epochs (step is "one batch", epoch is "one full pass through the whole data")
    logging_strategy="steps",
    eval_steps=500, #eval every 500 steps
    logging_steps=500,
    learning_rate=5e-5, #learning rate of the gradient descent
    max_steps=10000,
    load_best_model_at_end=True, #when done, load the best model you have (which is not necessarily the one after the last step)
    per_device_train_batch_size=16 #batch size
)

pprint(trainer_args)

In [ ]:
accuracy = evaluate.load("accuracy")

def compute_accuracy(outputs_and_labels):
    outputs, labels = outputs_and_labels
    predictions = np.argmax(outputs, axis=-1) #pick the index of the "winning" label among the outputs, i.e. argmax
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
# Make a new model, this will also initialize it
mlp = MLP(mlp_config)


# Argument gives the number of evaluation tries of patience before early stopping
# i.e. training is stopped when the evaluation loss fails to improve
# certain number of times the model is evaluated, in this case 5 consecutive times
early_stopping = transformers.EarlyStoppingCallback(5)

trainer = transformers.Trainer(
    model=mlp,
    args=trainer_args,
    train_dataset=dset_tokenized["train"],
    eval_dataset=dset_tokenized["test"].select(range(1000)), #make a smaller subset to evaluate on
    compute_metrics=compute_accuracy,
    data_collator=collator,
    callbacks=[early_stopping]
)

# FINALLY!
# a bit slow on CPU but doable in about 4min
# about 1.5x faster on GPU (this neural net is too simple to really make the CPU/GPU difference stand out)
trainer.train()

In [ ]:
mlp.save_pretrained("mlp-imdb2")

In [ ]:
mlp2=MLP.from_pretrained("mlp-imdb2")

In [ ]:
# The same trainer with slightly different arguments
# can be used to run prediction and get scores

eval_args = transformers.TrainingArguments(
  do_train=False,
  do_eval=False
)

trainer = transformers.Trainer(
    model=mlp2,
    args=eval_args,
    compute_metrics=compute_accuracy,
    data_collator=collator
)



In [ ]:
eval_results = trainer.predict(dset_tokenized["test"])
print(eval_results)
print('Accuracy:', eval_results.metrics['test_accuracy'])

Now my own part:

In [ ]:
weights=mlp.embedding.weight.detach().cpu().numpy()

Running the code to find nearest neighbor for the token 'study'. This could be interpreted as a noun or as a verb.

In [ ]:
qry_idx=tokenizer.vocab["study"] #embedding of "study"
idx2word={v:k for k,v in tokenizer.vocab.items()}

#calculate the distance of the  embedding to all other embeddings
distance_to_qry=metrics.pairwise.euclidean_distances(weights[qry_idx:qry_idx+1,:],weights)
nearest_neighbors=np.argsort(distance_to_qry) #indices of nearest words
print("*Top of the list:*")
for nearest in nearest_neighbors[0,:20]:
    print(idx2word[nearest])
print("\n(...)\n")
print("*Bottom of the list:*")
for nearest in nearest_neighbors[0,-20:]:
    print(idx2word[nearest])


Part B: